# 动态规划算法学习笔记

本文档基于《动手学强化学习》第 4 章“动态规划算法”整理，原文链接：<https://hrl.boyuai.com/chapter/1/%E5%8A%A8%E6%80%81%E8%A7%84%E5%88%92%E7%AE%97%E6%B3%95/>。

这次整理的目标不是把公式再抄一遍，而是把初学者最容易卡住的几个问题讲得更朴素一些：

- 动态规划在强化学习里到底在“算”什么？
- `P[state][action]`、`v`、`pi` 这些变量到底是什么结构？
- 策略迭代和价值迭代分别是怎么一步一步跑起来的？

如果你是第一次接触这一章，建议这样读：

1. 先看第 1 节，建立整体直觉。
2. 再看第 3 节，把 Cliff Walking 的地图、状态编号和 `P` 看懂。
3. 最后看第 4、5 节，把公式和代码对应起来。

## 1. 先不急着看公式：DP 在强化学习里到底在做什么？

动态规划（Dynamic Programming, DP）不是“靠和环境反复交互去试错学习”，而是建立在一个前提上：**环境的规则已经知道了**。

你可以把它想成手里已经拿到一本“环境说明书”。这本说明书会告诉你：

- 在状态 `s` 做动作 `a`，可能去哪些下一状态 `s'`
- 每种结果发生的概率是多少
- 这一步会拿到多少奖励

在这种前提下，DP 做的事情就很朴素：**既然规则都知道了，那就不要靠采样猜，直接把每个状态的长期回报算出来。**

读这一章时，先把下面 4 个词分清楚：

- **状态（state）**：智能体现在站在哪个格子、处于什么局面。
- **动作（action）**：在当前状态下可以做什么，比如上、下、左、右。
- **策略（policy）**：在每个状态下，你打算怎么选动作。它可以是确定的，也可以是按概率随机选。
- **价值函数（value）**：从当前状态出发，未来长期奖励的期望总和。

这里又有两个最常见的价值：

- **状态价值 `v(s)`**：站在状态 `s` 上，这个状态整体“值多少钱”。
- **动作价值 `q(s,a)`**：在状态 `s` 先做动作 `a`，这个动作“值多少钱”。

所以我会把 DP 概括成一句话：

> 已知环境规则后，反复利用贝尔曼方程，把每个状态或动作的长期价值算到稳定。

基于这条主线，后面两种算法就很好区分了：

- **策略迭代**：先把“当前策略到底值多少钱”算清楚，再按这个结果改策略。
- **价值迭代**：不把某个策略完整评估到底，直接朝着最优价值函数逼近。

## 2. 关键公式整理：每个公式到底在回答什么问题？

这一节的目标不是让人死记公式，而是让人知道：**每个公式到底在算什么，代码里又对应哪一步。**

### 2.1 策略评估：如果我坚持按照当前策略行动，这个状态值多少钱？

给定策略 $\pi$，状态价值函数满足贝尔曼期望方程：

$$
v_{\pi}(s)=\sum_a \pi(a\mid s) \sum_{s',r} p(s',r\mid s,a) \left[r + \gamma v_{\pi}(s')\right]
$$

可以把它翻译成一句人话：

> 状态 `s` 的价值 = 在当前策略下，所有动作价值的加权平均。

这里每个符号分别表示：

- `v_\pi(s)`：按照策略 `\pi` 行动时，状态 `s` 的长期回报。
- `\pi(a|s)`：在状态 `s` 下选择动作 `a` 的概率。
- `p(s', r | s, a)`：在状态 `s` 做动作 `a` 后，以什么概率到达 `s'` 并拿到奖励 `r`。
- `\gamma`：折扣因子，表示“未来奖励的重要程度”。

代码层面，这个公式通常会展开成三层意思：

1. 先枚举动作 `a`
2. 再枚举这个动作可能带来的转移结果 `(p, next_state, reward, done)`
3. 最后按策略概率把不同动作的价值加权求和

如果一个状态已经终止了，那么它后面就没有未来价值了。代码里常见的写法是乘上 `(1 - done)`，让终止状态的未来项自动变成 0。

### 2.2 策略提升：既然知道每个动作值多少钱，就别再把概率浪费在差动作上

如果已经得到 $v_{\pi}(s)$，就可以计算动作价值：

$$
q_{\pi}(s,a)=\sum_{s',r} p(s',r\mid s,a)\left[r + \gamma v_{\pi}(s')\right]
$$

这条公式回答的问题是：

> 在状态 `s` 下，如果我先强行做动作 `a`，然后再继续按当前价值函数看未来，这个动作值多少钱？

一旦 4 个动作的 `q(s,a)` 都算出来了，接下来就可以做贪心改进：

$$
\pi'(s)=\arg\max_a q_{\pi}(s,a)
$$

这句话的意思很朴素：**哪个动作价值最大，就让新策略更偏向哪个动作。**

如果多个动作并列第一，那么它们都算“最优动作”。这份 notebook 里的实现会让这些并列最优动作均分概率，而不是只硬选其中一个。

### 2.3 价值迭代：每轮都直接问“此刻最好的动作是谁？”

价值迭代直接使用贝尔曼最优方程：

$$
v_*(s)=\max_a \sum_{s',r} p(s',r\mid s,a)\left[r + \gamma v_*(s')\right]
$$

和策略迭代相比，它少了一层“先按当前策略加权平均”的过程，直接对动作取最大值。直观上就是：

- **策略迭代**：先完整评价一个策略，再改进它。
- **价值迭代**：每轮更新都直接把当前看起来最好的动作拿过来。

所以在代码里，你可以抓住一个最直观的区别：

- 看到 `sum(...)` 往往是在做策略评估
- 看到 `max(...)` 往往是在做价值迭代或策略提升

In [1]:
from pprint import pprint

## 3. 悬崖漫步环境：先把地图和数据结构看懂

原文先在一个 4x12 的网格世界里演示 DP。对于初学者来说，这一节最重要的不是立刻看代码，而是先把地图和编号方式看清楚。

如果把坐标写成 `(row, col)`，这个环境大致可以理解成：

```text
(row,col)
(0,0) ........................ (0,11)
(1,0) ........................ (1,11)
(2,0) ........................ (2,11)
S      C   C   C   C   C   C   C   C   C   C   G
```

这里：

- `S` 是起点，在左下角，也就是 `(3, 0)`
- `G` 是终点，在右下角，也就是 `(3, 11)`
- `C` 是悬崖，在底边中间 `(3, 1)` 到 `(3, 10)`
- 行号从上往下增大，列号从左往右增大

代码里不会一直保存 `(row, col)` 这种二维坐标，而是会把它压平成一个一维状态编号：

$$
state = row \times ncol + col
$$

在这个 4x12 的例子里：

- `(0,0) -> 0`
- `(1,0) -> 12`
- `(3,0) -> 36`
- `(3,11) -> 47`

下面环境代码真正要做的事，是提前把每个状态、每个动作的后果都写进 `P` 里。

`P` 的结构可以先记成：

```python
P[state][action] = [(prob, next_state, reward, done)]
```

也就是：

- 第一层 `state`：当前状态
- 第二层 `action`：在这个状态下选择哪个动作
- 第三层列表：这个动作可能带来的所有结果

因为 Cliff Walking 是确定性环境，所以第三层列表里通常只有一个元组；如果是随机环境，这里就可能有多个转移结果。

In [2]:
class CliffWalkingEnv:
    """悬崖漫步环境。状态编号按行优先展开。"""

    def __init__(self, ncol=12, nrow=4):
        self.ncol = ncol
        self.nrow = nrow
        self.P = self._create_transition_matrix()

    def _create_transition_matrix(self):
        P = [[[] for _ in range(4)] for _ in range(self.nrow * self.ncol)]
        change = [(0, -1), (0, 1), (-1, 0), (1, 0)]  # 上、下、左、右

        for i in range(self.nrow):
            for j in range(self.ncol):
                state = i * self.ncol + j
                for action, (dx, dy) in enumerate(change):
                    if i == self.nrow - 1 and j > 0:
                        P[state][action] = [(1.0, state, 0, True)]
                        continue

                    next_x = min(self.ncol - 1, max(0, j + dx))
                    next_y = min(self.nrow - 1, max(0, i + dy))
                    next_state = next_y * self.ncol + next_x
                    reward = -1
                    done = False

                    if next_y == self.nrow - 1 and next_x > 0:
                        done = True
                        if next_x != self.ncol - 1:
                            reward = -100

                    P[state][action] = [(1.0, next_state, reward, done)]
        return P


def print_agent(agent, action_meaning, disaster=None, end=None):
    disaster = set(disaster or [])
    end = set(end or [])

    print("状态价值：")
    for i in range(agent.env.nrow):
        row = []
        for j in range(agent.env.ncol):
            value = agent.v[i * agent.env.ncol + j]
            row.append(f"{value:6.3f}")
        print(" ".join(row))

    print("\n策略：")
    for i in range(agent.env.nrow):
        row = []
        for j in range(agent.env.ncol):
            state = i * agent.env.ncol + j
            if state in disaster:
                row.append("****")
            elif state in end:
                row.append("EEEE")
            else:
                probs = agent.pi[state]
                row.append("".join(action_meaning[k] if probs[k] > 0 else "o" for k in range(len(action_meaning))))
        print(" ".join(row))

### 3.1 读环境代码时，建议先盯住这 4 个量

下面那段 `CliffWalkingEnv` 代码不是在“玩一局游戏”，而是在**提前把整张地图的规则表算出来**。读的时候最值得盯住 4 个量：

- `i`：当前行号（row）
- `j`：当前列号（col）
- `next_x / next_y`：执行动作后，下一个格子的列号和行号
- `next_state = next_y * ncol + next_x`：把二维坐标再压回一维状态编号

最外层这句：

```python
P = [[[] for _ in range(4)] for _ in range(self.nrow * self.ncol)]
```

可以先粗暴理解成：**先准备一张大表，这张表有“状态数”行，每行 4 个动作槽位。**

后面三层循环做的事，就是对每个状态、每个动作，依次填进去“会去哪里、奖励多少、是否结束”。

如果看到下面这句：

```python
next_x = min(self.ncol - 1, max(0, j + dx))
```

不要被它吓住。它只是表示：**先按动作移动一步，如果越界了，就卡在边界上。**

所以这段环境代码本质上不是在做复杂数学，而是在做一件很朴素的事：**把地图规则全部预先写成一个可查询的数据表。**

## 4. 策略迭代：先评价当前策略，再把策略改得更聪明

在看代码前，先把里面几个最容易混的对象记住：

- `self.v`：一维列表，长度等于状态数，`self.v[s]` 表示状态 `s` 的价值。
- `self.pi`：二维列表，形状大致是 `状态数 x 4`，`self.pi[s][a]` 表示在状态 `s` 下选择动作 `a` 的概率。
- `_calc_qsa(s, a)`：一个小工具函数，用来计算单个动作价值 `Q(s,a)`。

整套策略迭代代码其实就做三件事：

1. `policy_evaluation`：固定当前策略 `pi`，把每个状态值 `v(s)` 算出来。
2. `policy_improvement`：对每个状态把 4 个动作的 `Q(s,a)` 算出来，然后把策略改得更偏向最大动作。
3. `policy_iteration`：不停交替执行上面两步，直到策略不再变化。

你可以把整个流程记成一句话：

> 固定策略 -> 计算状态价值 -> 用状态价值算动作价值 -> 把策略改得更贪心 -> 重复

这里还有一个实现细节很重要：如果多个动作并列最优，就让它们均分概率。这样做的好处是更忠实地反映“最优策略可能不唯一”。

In [3]:
class PolicyIteration:
    def __init__(self, env, theta=1e-3, gamma=0.9):
        self.env = env
        self.theta = theta
        self.gamma = gamma
        self.v = [0.0] * (env.ncol * env.nrow)
        self.pi = [[0.25, 0.25, 0.25, 0.25] for _ in range(env.ncol * env.nrow)]

    def _calc_qsa(self, state, action):
        qsa = 0.0
        for p, next_state, reward, done in self.env.P[state][action]:
            qsa += p * (reward + self.gamma * self.v[next_state] * (1 - done))
        return qsa

    def policy_evaluation(self):
        evaluation_rounds = 0
        while True:
            max_diff = 0.0
            new_v = [0.0] * (self.env.ncol * self.env.nrow)

            for s in range(self.env.ncol * self.env.nrow):
                state_value = 0.0
                for a in range(4):
                    state_value += self.pi[s][a] * self._calc_qsa(s, a)
                new_v[s] = state_value
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))

            self.v = new_v
            evaluation_rounds += 1
            if max_diff < self.theta:
                break
        return evaluation_rounds

    def policy_improvement(self):
        policy_stable = True
        for s in range(self.env.nrow * self.env.ncol):
            old_action_distribution = self.pi[s][:]
            qsa_list = [self._calc_qsa(s, a) for a in range(4)]
            max_q = max(qsa_list)
            max_count = qsa_list.count(max_q)
            self.pi[s] = [1 / max_count if q == max_q else 0 for q in qsa_list]
            if self.pi[s] != old_action_distribution:
                policy_stable = False
        return policy_stable

    def policy_iteration(self, verbose=True):
        history = []
        while True:
            rounds = self.policy_evaluation()
            stable = self.policy_improvement()
            history.append(rounds)
            if verbose:
                print(f"第 {len(history)} 次策略评估共迭代 {rounds} 轮")
            if stable:
                break
        return history

### 4.1 把三段核心函数翻译成人话

看到 `PolicyIteration` 这段代码时，可以不急着逐字符读，先抓住每个函数在回答什么问题：

- `policy_evaluation`：**如果我继续按当前策略走，每个状态值多少钱？**
- `policy_improvement`：**既然每个动作的价值都能算出来，那我为什么还要给差动作概率？**
- `policy_iteration`：**把前两步反复做，直到策略不再变化。**

这份实现里有一个很实用的小改写：把计算动作价值的逻辑单独提成了 `_calc_qsa(s, a)`。这样读主流程时，可以把它直接理解成“求 `Q(s,a)`”。

所以 `policy_evaluation` 里的核心语句：

```python
state_value += self.pi[s][a] * self._calc_qsa(s, a)
```

翻译成人话就是：

> 当前状态的价值，等于 4 个动作价值按策略概率做加权平均。

而 `policy_improvement` 里的核心语句：

```python
qsa_list = [self._calc_qsa(s, a) for a in range(4)]
max_q = max(qsa_list)
```

翻译成人话就是：

> 先把 4 个动作都算一遍，然后把最好的动作挑出来。

这里的“提升”不是手动把数值改大，而是让策略变得**更偏向高价值动作**。等这个新策略再经过一轮评估后，状态价值通常就会更好。

In [4]:
cliff_env = CliffWalkingEnv()
action_meaning = ["^", "v", "<", ">"]

pi_agent = PolicyIteration(cliff_env, theta=1e-3, gamma=0.9)
pi_history = pi_agent.policy_iteration()

print("每次策略评估的迭代轮数:", pi_history)
print_agent(pi_agent, action_meaning, disaster=range(37, 47), end=[47])

第 1 次策略评估共迭代 60 轮
第 2 次策略评估共迭代 72 轮
第 3 次策略评估共迭代 44 轮
第 4 次策略评估共迭代 12 轮
第 5 次策略评估共迭代 1 轮
每次策略评估的迭代轮数: [60, 72, 44, 12, 1]
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000
-7.458  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000

策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo
^ooo **** **** **** **** **** **** **** **** **** **** EEEE


### 策略迭代实验后的记录：输出到底该怎么读？

跑完上面的代码后，建议先不要急着只看“策略对不对”，而是先学会读输出。

`print_agent` 会打印两部分内容：

- **状态价值表**：每个格子的数值就是 `v(s)`，表示从这里出发、按当前策略继续走时的长期回报。
- **策略表**：每个格子的箭头表示这个状态下的最优动作。

策略表里有几个符号约定：

- `****`：悬崖位置
- `EEEE`：终点
- `o`：这个方向不是当前策略会选的方向
- `^ v < >`：这四个位置分别对应上、下、左、右

如果某个格子里同时出现多个箭头，说明这些动作并列最优，所以策略把概率分给了多个方向。

在这个环境里，最终策略通常会沿着悬崖上方一行向右走，而不是贴着悬崖冲过去。原因不是“几何距离更短就一定更好”，而是价值函数会同时考虑：

- 每走一步的 `-1`
- 掉进悬崖的 `-100`
- 终止后不再有未来回报

所以价值函数学到的不是“离终点还有几步”，而是**未来所有奖励和风险综合起来后的长期期望**。这也是 DP 比“肉眼看最短路”更可靠的地方。

## 5. 价值迭代：每轮都直接问“当前最好的动作是谁？”

如果说策略迭代是“先把一个策略看清楚，再改策略”，那么价值迭代更像是：

> 不等待某个策略被完整评估到底，而是每一轮都直接把当前最好的动作拿过来。

它和策略迭代共用同一个环境、同一个 `Q(s,a)` 计算方式，最大的区别只在更新状态价值时这一步：

- 策略迭代会做加权平均：`V(s) = sum_a pi(a|s) Q(s,a)`
- 价值迭代会直接取最大：`V(s) = max_a Q(s,a)`

所以从代码视角看，价值迭代最关键的一句就是：

```python
new_v[s] = max(qsa_list)
```

这相当于把“策略评估 + 策略提升”压缩到了一轮更新里。

这里还有一个初学者很容易疑惑的点：为什么价值迭代跑完后，还要再调用一次 `get_policy()`？

原因是价值迭代在主循环里主要维护的是 **最优价值函数的近似值 `v`**，并没有像策略迭代那样始终显式维护一份稳定的 `pi`。所以等 `v` 收敛后，还需要再根据 `v` 对每个状态做一次贪心选择，把策略恢复出来。

In [5]:
class ValueIteration:
    def __init__(self, env, theta=1e-3, gamma=0.9):
        self.env = env
        self.theta = theta
        self.gamma = gamma
        self.v = [0.0] * (env.ncol * env.nrow)
        self.pi = [None for _ in range(env.ncol * env.nrow)]

    def _calc_qsa(self, state, action):
        qsa = 0.0
        for p, next_state, reward, done in self.env.P[state][action]:
            qsa += p * (reward + self.gamma * self.v[next_state] * (1 - done))
        return qsa

    def value_iteration(self, verbose=True):
        iteration = 0
        while True:
            max_diff = 0.0
            new_v = [0.0] * (self.env.ncol * self.env.nrow)

            for s in range(self.env.ncol * self.env.nrow):
                qsa_list = [self._calc_qsa(s, a) for a in range(4)]
                new_v[s] = max(qsa_list)
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))

            self.v = new_v
            iteration += 1
            if max_diff < self.theta:
                break

        if verbose:
            print(f"价值迭代共进行了 {iteration} 轮")
        self.get_policy()
        return iteration

    def get_policy(self):
        for s in range(self.env.nrow * self.env.ncol):
            qsa_list = [self._calc_qsa(s, a) for a in range(4)]
            max_q = max(qsa_list)
            max_count = qsa_list.count(max_q)
            self.pi[s] = [1 / max_count if q == max_q else 0 for q in qsa_list]

In [6]:
vi_agent = ValueIteration(cliff_env, theta=1e-3, gamma=0.9)
vi_rounds = vi_agent.value_iteration()

print_agent(vi_agent, action_meaning, disaster=range(37, 47), end=[47])
print("\n两种方法的价值函数是否一致:", all(abs(a - b) < 1e-6 for a, b in zip(pi_agent.v, vi_agent.v)))
print("两种方法的策略是否一致:", pi_agent.pi == vi_agent.pi)

价值迭代共进行了 15 轮
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000
-7.458  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000

策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo
^ooo **** **** **** **** **** **** **** **** **** **** EEEE

两种方法的价值函数是否一致: True
两种方法的策略是否一致: True


### 我对两种算法差异的总结

| 对比点 | 策略迭代 | 价值迭代 |
| --- | --- | --- |
| 每轮在做什么 | 先评估当前策略，再改进策略 | 直接按当前最大动作更新价值 |
| 是否显式维护策略 | 是，整个过程中一直有 `pi` | 否，最后再从 `v` 恢复 |
| 核心更新 | `sum_a pi(a|s) Q(s,a)` | `max_a Q(s,a)` |
| 单次迭代代价 | 通常更大，因为评估阶段会多轮迭代 | 通常更小，更直接 |
| 直觉 | “先看清楚，再行动” | “边估边选当前最优” |
| 停止条件 | 策略不再变化 | 价值函数变化足够小 |

在 Cliff Walking 这种小环境里，两者最后会得到相同的最优价值和最优策略；差别主要体现在“中间怎么走到这个结果”。

所以初学时，可以把它们记成：

- **策略迭代**：更像在反复打磨一份策略
- **价值迭代**：更像在直接逼近最优价值函数

## 6. FrozenLake：一旦转移变随机，最优策略就不只看几何距离

原文还用 FrozenLake 做了一个很好的补充。这个环境和悬崖漫步的最大区别是：**动作不是确定性的**。智能体在冰面上可能会滑到别的位置，因此最优动作未必是“几何上最短”的动作。

这也是前面 `P[state][action]` 要设计成“列表里放多个转移结果”的原因。

在 Cliff Walking 里，某个动作通常只有一个结果，例如：

```python
P[s][a] = [(1.0, next_state, reward, done)]
```

但在 FrozenLake 里，一个动作可能对应多个结果，例如：

```python
P[s][a] = [
    (0.33, s1, r1, done1),
    (0.33, s2, r2, done2),
    (0.33, s3, r3, done3),
]
```

所以在接近终点的位置，看起来“向右”最合理，但如果向右后有较大概率滑进冰洞，那么“更保守”的方向反而可能更优。

这正好说明：**DP 真正依赖的是环境模型 `P`，而不是地图长得像不像最短路问题。**

下面这段代码会优先尝试 `gymnasium`，再尝试 `gym`。如果本地没装相关包，它会自动跳过。原文使用的是 `FrozenLake-v0`，但在较新的环境里通常需要 `FrozenLake-v1`。

In [7]:
def make_frozen_lake_env():
    gym_module = None
    env = None
    env_name = None

    try:
        import gymnasium as gym_module
        for candidate in ("FrozenLake-v1", "FrozenLake-v0"):
            try:
                env = gym_module.make(candidate)
                env_name = candidate
                break
            except Exception:
                pass
    except ImportError:
        try:
            import gym as gym_module
            for candidate in ("FrozenLake-v1", "FrozenLake-v0"):
                try:
                    env = gym_module.make(candidate)
                    env_name = candidate
                    break
                except Exception:
                    pass
        except ImportError:
            return None, None

    if env is None:
        return None, None

    env = env.unwrapped
    if hasattr(env, "desc"):
        env.nrow, env.ncol = env.desc.shape
    return env, env_name


frozen_env, frozen_env_name = make_frozen_lake_env()
if frozen_env is None:
    print("当前环境没有安装 gym/gymnasium，FrozenLake 示例先保留代码，不执行。")
else:
    holes = set()
    ends = set()
    for s in frozen_env.P:
        for a in frozen_env.P[s]:
            for p, next_state, reward, done in frozen_env.P[s][a]:
                if reward == 1.0:
                    ends.add(next_state)
                if done:
                    holes.add(next_state)
    holes -= ends

    print("FrozenLake 环境版本:", frozen_env_name)
    print("冰洞状态:", holes)
    print("目标状态:", ends)
    print("状态 14 的转移信息:")
    pprint(frozen_env.P[14])

当前环境没有安装 gym/gymnasium，FrozenLake 示例先保留代码，不执行。


In [8]:
if frozen_env is not None:
    frozen_action_meaning = ["<", "v", ">", "^"]

    frozen_pi_agent = PolicyIteration(frozen_env, theta=1e-5, gamma=0.9)
    frozen_pi_agent.policy_iteration(verbose=False)
    print("策略迭代结果：")
    print_agent(frozen_pi_agent, frozen_action_meaning, disaster=holes, end=ends)

    frozen_vi_agent = ValueIteration(frozen_env, theta=1e-5, gamma=0.9)
    frozen_vi_agent.value_iteration(verbose=False)
    print("\n价值迭代结果：")
    print_agent(frozen_vi_agent, frozen_action_meaning, disaster=holes, end=ends)

## 7. 最后的总结：学完这一章，最该带走什么？

如果把这一章压缩成几句最值得带走的话，我会记住下面几点：

- DP 不是靠采样“猜”出来的，而是在已知环境模型时直接“算”出来的。
- `状态价值` 关注的是“站在这里值多少钱”，`动作价值` 关注的是“先做这个动作值多少钱”。
- 策略迭代的主线是：先评估当前策略，再把策略改得更好。
- 价值迭代的主线是：每一轮都直接朝最优价值函数逼近。
- 确定性环境更符合几何直觉，随机环境则会把风险显式纳入最优策略。

如果后面继续学时序差分、Q-learning 或 DQN，可以把它们和这一章对照着看：那些方法本质上是在**不知道完整环境模型时，用采样去近似完成这里的事情**。